# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 2: Transformer Architecture - Encoder-only Models</font>

# <font color="#003660">Notebook 4: Named Entity Recognition with Transformers</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>
<center>
<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... understand the differences between sequence and token classification, <br>
        ... know how to fine-tune a NER model on labelled data, and <br>
        ... know how to share a model on the Hugging Face Hub (optional).
    </font>
</div>
</center>
</p>

The following content is heavily inspired by the following excellent sources:


*   Tunstall et al. (2022): Natural Language Processing with Transformers. O'Reilly. https://www.oreilly.com/library/view/natural-language-processing/9781098103231/
*   Hugging Face: LLM Course. https://huggingface.co/learn/llm-course



# Token vs. Sequence Classification

Token classification assigns a label to individual tokens in a sentence. One of the most common token classification tasks is Named Entity Recognition (NER). NER attempts to find a label for each entity in a sentence, such as a person, location, or organization. Learn more: https://huggingface.co/learn/llm-course/chapter7/2

The figures below show the main differences of using encoders like BERT and its siblings and cousins for sequence and token classification. When doing **sequence classification** (e.g., sentiment analysis), we feed a sequence into the model and only work with the contextual embedding of the [CLS] token when training the classification head of the model. In contrast, when doing **token classification** we feed the contextual embeddings of all tokens through the classification head of the model. 

<center><img width=500 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/seq_class.png"/><br></center>

<hr>

<center><img width=500 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/token_class.png"/><br></center>

# Import Packages

As always, we first need to load a number of required Python packages:
- `pandas` provides high-performance, easy-to-use data structures and data analysis tools.
- `numpy` is a library adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays.
- `torch` (PyTorch) is the open source deep learning library on which the `transformers` library is built. 
- `transformers` provides thousands of pretrained models (e.g., BERT, DistilBERT, RoBERTa, GPT) from the Hugging Face model hub, together with tools to use and fine-tune them.
- `datasets` and `evaluate` are libraries from Hugging Face to feed Transformers with data and evaluate their predictive accuracy.

In [ ]:
%pip install -q evaluate seqeval

In [ ]:
import numpy as np
import torch
from huggingface_hub import notebook_login
from transformers import AutoTokenizer
from transformers import DataCollatorForTokenClassification
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import pipeline
from datasets import load_dataset
import evaluate

# Load data

In this notebook we will use the WNUT 17 dataset, which is a NER dataset focusing on emerging and rare entities. You can find more information about the dataset here: https://github.com/leondz/emerging_entities_17 and https://huggingface.co/datasets/manu/wnut_17

In [ ]:
wnut = load_dataset("manu/wnut_17")

In [ ]:
wnut["train"][0]

Each number in `ner_tags` represents a type of named entity (e.g., location, person). We can convert the numbers to textual labels to learn more about those entities.

In [ ]:
label_list = wnut["train"].features["ner_tags"].feature.names
label_list

The prefixes of the tags indicate whether a given token signifies the beginning or middle/end of a named entity:

* **B-**: indicates the beginning of an entity.
* **I-**: indicates a token is contained inside the same entity (for example, the State token is a part of an entity like Empire State Building).
* **O**: indicates that the token doesn’t correspond to any entity.

# Preprocess data

As always, the first thing to do when processing raw texts with Transformers is to tokenize the sequences. First, we need to load a tokenizer that is compatible with the architecture we want to use (here: DistilBERT, as in Notebooks 2 and 3).

Note that `distilbert-base-uncased` lowercases all texts. You might expect that a *cased* model works better for NER, as capitalization is an important clue ("Apple" the company vs. "apple" the fruit). But in tweets, capitalization is used very inconsistently, so it is not obvious whether a cased model helps here. We stick with the uncased model from Notebooks 2 and 3; you can compare it with the cased one in the exercises at the end.

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

Let's re-join one example from the training set and feed it through the tokenizer. This mimics how we would use the tokenizer on new data. 

In [ ]:
tokenized_input = tokenizer(" ".join(wnut["train"][0]["tokens"]))
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
tokens

As expected, the tokenizer performed sub-word tokenization, splitting, for example, `ESB` into `es` and `##b` (note the lowercasing). In addition, the tokenizer added the usual `[CLS]` and `[SEP]` tokens.

As a result, the tokenized sequence and the labels of our training data (which has NOT been tokenized with sub-word tokenization) are not aligned anymore. A quick look at the length of the two sequences confirms this.

In [ ]:
len(tokens)

In [ ]:
len(wnut["train"][0]["tokens"])

Hence, we need to realign the tokens and labels using the following process:

1. Map all tokens to their corresponding word with the `word_ids` method.
2. Assign the label `-100` to the special tokens `[CLS]` and `[SEP]`, so that they can be ignored by the loss function.
3. Only label the first token of a given word. Assign `-100` to other subtokens from the same word.

Below is a function to tokenize the text and, afterwards, realign the tokens and labels.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

Apply the function to the whole dataset (i.e., train, validation, and test).

In [ ]:
tokenized_wnut = wnut.map(tokenize_and_align_labels, batched=True)

Check whether the sequences are aligned now.

In [ ]:
tokenized_wnut

In [ ]:
tokenized_wnut["train"][0]

In [ ]:
len(tokens) == len(tokenized_wnut["train"][0]["input_ids"])

To preprocess our texts on-the-fly while training our model, we can use a data collator (more information: https://huggingface.co/docs/transformers/main_classes/data_collator).

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Fine-tune model

Now we are (almost) ready to fine-tune a pre-trained DistilBERT on our emerging and rare named entities dataset. At the end of the notebook, you will also see how you could publish your model (checkpoint + tokenizer) on the Hugging Face Hub.

As always, when training a model we need a compute_metrics function to calculate and display selected accuracy metrics during training. Here we use precision, recall, F1, and accuracy (they are all saved together in the seqeval evaluation object).

In [ ]:
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In order to be able to interpret the model's predictions, we also need two dictionaries to translate between the textual labels and their indices. We can create them directly from the list of labels.

In [ ]:
id2label = dict(enumerate(label_list))
label2id = {label: i for i, label in enumerate(label_list)}
id2label

🚀 🚀 🚀 Now we are really ready to fine-tune... 🚀 🚀 🚀

Load the pre-trained model, configure the classification head (how many labels?), and pass the dictionaries to translate between label IDs and textual labels.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id
)

Configure the Trainer. Here, we will only train for 2 epochs. In reality, you probably want to train longer!

In [ ]:
training_args = TrainingArguments(
    output_dir="my_awesome_wnut_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    push_to_hub=False, # You need to be logged in for this
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_wnut["train"],
    eval_dataset=tokenized_wnut["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Go! 🏁

In [ ]:
trainer.train()

The Trainer evaluated the model on the validation set after each epoch and loaded the best one. For a final, unbiased estimate of its performance, evaluate it on the test set. Note that the F1-score is computed per *entity*: an entity only counts as correct if all of its tokens get the right labels. Don't be disappointed by the low score: WNUT 17 is deliberately hard, as it focuses on rare and emerging entities in noisy tweets, most of which the model has never seen before.

In [ ]:
trainer.evaluate(tokenized_wnut["test"])

# Inference

## Using a Pipeline

In the following cells, we instantiate a pipeline, which integrates a trained model (aka checkpoint) and compatible tokenizer, and feed it with a sample sentence. With `aggregation_strategy="first"`, the pipeline merges sub-word tokens into whole words and consecutive B- and I- tokens into entities (as in Notebook 1).

In [ ]:
text = "The Golden State Warriors are an American professional basketball team based in San Francisco."

In [ ]:
ner_classifier = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="first")

In [ ]:
ner_classifier(text)

## By Hand

Instead of using a pipeline, we can also compute predictions step-by-step, token by token. Reproducing each of the steps by hand ✍️ will increase your understanding of the underlying logic.

Tokenize the example sentence.

In [ ]:
inputs = tokenizer(text, return_tensors="pt")
inputs

Do a forward pass through the network.

In [ ]:
with torch.no_grad():
    # Move inputs to the same device as the model
    inputs = {name: tensor.to(model.device) for name, tensor in inputs.items()}
    logits = model(**inputs).logits
logits

For each token, get the index of the label with the highest score.

In [ ]:
predictions = torch.argmax(logits, dim=2)
predictions

Translate the indices to textual labels.

In [ ]:
predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]
predicted_token_class

# Share Your Model on the Hugging Face Hub (Optional)

Fine-tuned models can be shared with others via the Hugging Face Hub, just like the pre-trained models we downloaded. For this, you need a (free) Hugging Face account and an access token with write permission (https://huggingface.co/settings/tokens). Uncomment the following lines to log in and upload your model. Afterwards, anybody can use it with `pipeline("ner", model="<your-username>/my_awesome_wnut_model")`.

In [ ]:
# notebook_login()
# trainer.push_to_hub()

# Exercises

1. **Cased vs. uncased:** Fine-tune `distilbert-base-cased` instead of `distilbert-base-uncased` (change `model_name`). Does capitalization help or hurt on tweets? Careful: with this small dataset, the F1-score can differ by several points between two runs of the *same* model (in our test runs, between 0.44 and 0.53 on the validation set). Train each model a few times before you draw conclusions.
2. **Training time:** Train for more epochs (e.g., 5). How do precision and recall develop?
3. **Per-entity results:** `seqeval.compute()` also returns precision, recall, and F1 for each entity type. Which entity types are hardest to recognize, and why?
4. **Your own text:** Apply the model to a sentence of your choice, e.g., a recent news headline. Which entities does it find, and which does it miss?

In [ ]:
# YOUR CODE GOES HERE